# Paper-Ready Table and Figure Exports

This notebook rebuilds the manuscript tables and figures from generated repo outputs. It writes paste-ready Markdown, Word, LaTeX, SVG, and CSV files. It does not change the analysis panel.

## Setup

Run this notebook from the repository root or from the `notebooks/` directory. The code adds `src/` to the Python path so the notebook uses the local project code.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, SVG, display

# Find the repository folder whether this notebook is opened from the root or notebooks/.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Tell Python to use the local project code instead of any installed package with the same name.
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# These functions rebuild exports from the current analysis outputs.
from coa_finaid_subs.descstat_tables import build_descstat_tables, format_export_table
from coa_finaid_subs.estimate_tables import build_estimate_tables
from coa_finaid_subs.report_figures import build_report_figures

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
REPO

## Descriptive Tables

The manuscript overview uses one display mean and standard deviation per variable. The winsorization audit is split out so the main table stays readable while the appendix keeps the raw-versus-capped evidence.

In [ ]:
# The descriptive tables start from the prepared public/private nonprofit analysis panel.
PANEL = REPO / "outputs" / "analysis_panel" / "public_private_nonprofit" / "analysis_panel_coa_headroom_2009_2023_public_private_nonprofit.parquet"

# The config file controls variable labels, sections, and display-only winsorization caps.
DESC_CONFIG = REPO / "config" / "descstat_variables.csv"
DESC_OUTPUT = REPO / "outputs" / "descriptive_tables"

# This writes overview, section-level, winsorization-audit, and appendix tables.
desc_paths = build_descstat_tables(
    input_panel=PANEL,
    output_dir=DESC_OUTPUT,
    config=DESC_CONFIG,
    scope_label="public_private_nonprofit",
)
desc_paths

### Manuscript Overview

Use this as the short descriptive-statistics table if the paper has space for only one descriptive table.

In [ ]:
# Load the compact manuscript table and format numbers for screen review.
desc_overview = pd.read_csv(desc_paths["paper_csv"])
display(format_export_table(desc_overview))

### Section Tables

Use these smaller tables when the paper reads better with separate COA, aid, finance, and scale blocks.

In [ ]:
# Show every section table separately. Each section is also exported to CSV, Markdown, LaTeX, and Word.
section_csv_paths = {key: path for key, path in desc_paths.items() if key.startswith("section_") and key.endswith("_csv")}
for key, path in section_csv_paths.items():
    title = key.removeprefix("section_").removesuffix("_csv").replace("_", " ").title()
    display(Markdown(f"#### {title}"))
    display(format_export_table(pd.read_csv(path)))

### Winsorization Audit

This table belongs in the appendix. It documents how much the display-only caps change means and standard deviations.

In [ ]:
# The audit keeps raw and winsorized statistics side by side without crowding the manuscript table.
winsor_audit = pd.read_csv(desc_paths["winsor_audit_csv"])
display(format_export_table(winsor_audit))

## Estimate Tables

The estimates are split into readable blocks. The full appendix export keeps the complete set of focal, interaction, and component rows.

In [ ]:
# The fixed-effects tables start from generated estimates, not hand-entered numbers.
FE_DIR = REPO / "outputs" / "fixed_effects"
EST_OUTPUT = REPO / "outputs" / "estimate_tables"

# This writes the main, aid-outcome, sector, robustness, component, and appendix tables.
estimate_paths = build_estimate_tables(
    fixed_effects_dir=FE_DIR,
    output_dir=EST_OUTPUT,
)
estimate_paths

### Main Institutional-Grant Estimates

This is the main paper table for the institutional-grant result.

In [ ]:
main_estimates = pd.read_csv(estimate_paths["paper_csv"])
display(main_estimates)

### Aid-Outcome Diagnostics

These estimates show whether the same headroom measures move Pell, institutional-grant, and federal-loan outcomes in the expected places.

In [ ]:
aid_outcomes = pd.read_csv(estimate_paths["aid_outcomes_csv"])
display(aid_outcomes)

### Sector and Robustness Checks

These tables separate sector patterns and specification checks from the main result.

In [ ]:
for label, key in [
    ("Sector checks", "sector_checks_csv"),
    ("Robustness checks", "robustness_csv"),
    ("COA component checks", "components_csv"),
]:
    display(Markdown(f"#### {label}"))
    display(pd.read_csv(estimate_paths[key]))

## Figures

The figures are intended for the paper or presentation slides. Each SVG has a matching CSV file with the plotted data.

In [ ]:
FIG_OUTPUT = REPO / "outputs" / "figures"
figure_paths = build_report_figures(
    analysis_dir=REPO / "outputs" / "analysis_panel" / "public_private_nonprofit",
    headroom_dir=REPO / "outputs" / "headroom_measures",
    fixed_effects_dir=FE_DIR,
    output_dir=FIG_OUTPUT,
)
figure_paths

In [ ]:
# Display the three figure SVGs inline for review.
for key in ["sample_counts_svg", "headroom_trends_svg", "main_estimate_forest_svg"]:
    display(SVG(filename=str(figure_paths[key])))

## Export Paths

Use Word files for direct copy/paste into Word or Google Docs. Use LaTeX files in a manuscript with `booktabs` and `longtable` loaded. Use SVG files for figures and CSV files for verification.

In [ ]:
# Collect export paths so they are easy to find after the notebook runs.
export_rows = []
for label, paths in [("descriptive", desc_paths), ("fixed_effects", estimate_paths), ("figures", figure_paths)]:
    for key, path in paths.items():
        if key == "summary" or key.endswith(("_md", "_tex", "_docx", "_csv", "_svg")):
            export_rows.append({"table_or_figure_set": label, "format": key, "path": str(path)})
exports = pd.DataFrame(export_rows)
display(exports)